<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-03/blob/main/hw_3_data__loading_sql_dql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 3 — Data Loading and SQL DQL

## 1. Setup and Data Loading

In [4]:
!pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [5]:
from ucimlrepo import fetch_ucirepo

# UCI Seoul Bike Sharing Demand
dataset = fetch_ucirepo(id=560)

df = pd.concat(
    [dataset.data.features, dataset.data.targets],
    axis=1
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (8760, 14)

Columns:
['Date', 'Rented Bike Count', 'Hour', 'Temperature', 'Humidity', 'Wind speed', 'Visibility', 'Dew point temperature', 'Solar Radiation', 'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning Day']


,Date,Rented Bike Count,Hour,Temperature,Humidity,Wind speed,Visibility,Dew point temperature,Solar Radiation,Rainfall,Snowfall,Seasons,Holiday,Functioning Day
0,1/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,1/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,1/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,1/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,1/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [6]:
csv_path = "/content/hw3_seoul_bike.csv"

df.to_csv(csv_path, index=False)

print("CSV saved:", csv_path)
print("Rows:", len(df))

CSV saved: /content/hw3_seoul_bike.csv
Rows: 8760


In [8]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS hw3_bike_clean CASCADE;
        DROP TABLE IF EXISTS hw3_bike_staging CASCADE;

        CREATE TABLE hw3_bike_staging (
            date_raw TEXT,
            rented_bike_count_raw TEXT,
            hour_raw TEXT,
            temperature_raw TEXT,
            humidity_raw TEXT,
            wind_speed_raw TEXT,
            visibility_raw TEXT,
            dew_point_temperature_raw TEXT,
            solar_radiation_raw TEXT,
            rainfall_raw TEXT,
            snowfall_raw TEXT,
            seasons_raw TEXT,
            holiday_raw TEXT,
            functioning_day_raw TEXT
        );
    """))

print("Staging table created successfully.")

Staging table created successfully.


In [9]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            cur.copy_expert(
                """
                COPY hw3_bike_staging (
                    date_raw,
                    rented_bike_count_raw,
                    hour_raw,
                    temperature_raw,
                    humidity_raw,
                    wind_speed_raw,
                    visibility_raw,
                    dew_point_temperature_raw,
                    solar_radiation_raw,
                    rainfall_raw,
                    snowfall_raw,
                    seasons_raw,
                    holiday_raw,
                    functioning_day_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                );
                """,
                file
            )

    raw_conn.commit()

finally:
    raw_conn.close()

print("CSV loaded into staging successfully.")

CSV loaded into staging successfully.


In [13]:
pd.read_sql("""
    SELECT COUNT(*) AS loaded_rows
    FROM hw3_bike_staging;
""", engine)

,loaded_rows
0,8760


In [15]:
pd.read_sql("""
    SELECT
        date_raw,
        rented_bike_count_raw,
        hour_raw,
        temperature_raw,
        seasons_raw,
        holiday_raw,
        functioning_day_raw
    FROM hw3_bike_staging
    LIMIT 5;
""", engine)

,date_raw,rented_bike_count_raw,hour_raw,temperature_raw,seasons_raw,holiday_raw,functioning_day_raw
0,1/12/2017,254,0,-5.2,Winter,No Holiday,Yes
1,1/12/2017,204,1,-5.5,Winter,No Holiday,Yes
2,1/12/2017,173,2,-6.0,Winter,No Holiday,Yes
3,1/12/2017,107,3,-6.2,Winter,No Holiday,Yes
4,1/12/2017,78,4,-6.0,Winter,No Holiday,Yes


## 2. Staging and Typed Table